# Buchwald-Hartwig core grid: GP-BO baseline vs. every candidate-level agent architecture

Runs the six-method "core grid" on the Buchwald-Hartwig C-N cross-coupling
benchmark: **GP-BO** (no agent), **SingleAgent**, **PCV** (Proposer-Critic-Verifier),
and three **BatchSelect** variants (LLM selector / random / top-acquisition
control conditions), using `qwen3:30b` served locally through Ollama.

10 seeds x 30 iterations x 6 methods. Writes `seeds_loop_checkpoint_qwen30b.pkl`,
which feeds Figure 8 (model-scale comparison) via `scripts/generate_figures.py`.
Also reads two comparison baselines produced by an earlier stage of the
project, `seeds_loop_checkpoint.pkl` and `seeds_loop_checkpoint_run2.pkl` --
see `experiments/README.md` for their provenance.

Renamed from the original research-log notebook `dia10.ipynb`. The shared
data loading, GP-BO proposal, and Proposer-Critic-Verifier/BatchSelect
agent logic used here live in `bayesllm.buchwald_hartwig.BuchwaldHartwigBenchmark`
(the `bayesllm` package in `src/`) -- every method there is a verified,
behaviour-preserving extraction of code that was duplicated identically
across this notebook and two others (`ucb_vs_ei_bh.ipynb`, `kernel_prior_bh.ipynb`);
see `VALIDATION.md` for what was checked.

In [ ]:
from bayesllm.buchwald_hartwig import BuchwaldHartwigBenchmark

bench = BuchwaldHartwigBenchmark(
    data_path="data/Dreher_and_Doyle_input_data.xlsx",
    model="qwen3:30b",
)
print(f"Feature space dimensionality: {bench.n_dims}")
print(f"Emulator in-sample R2 (sanity check only): {bench.emulator.score(bench.X_scaled, bench.y_raw):.3f}")

## Method-specific orchestration

The six methods share everything above via `bench`; each needs only a thin per-iteration wrapper.

In [ ]:
def run_pcv_iteration(X_bo, Y_bo, historial, bounds=None, max_rechazos=3, iteracion=0, verbose=False):
    """One PCV iteration: BO (q=1 LogEI) proposes a candidate, then
    Proposer -> Critic -> Verifier deliberation."""
    candidato_tensor = bench.propose_candidate(X_bo, Y_bo, bounds)
    return bench.run_pcv_deliberation(candidato_tensor, X_bo, Y_bo, historial,
                                       bounds=bounds, max_rechazos=max_rechazos,
                                       iteracion=iteracion, verbose=verbose)


def run_batch_iteration(X_bo, Y_bo, historial, bounds=None, q=4,
                         modo_seleccion='llm', max_rechazos=3, iteracion=0,
                         verbose=False):
    """
    One iteration of the batch method: qLogNEI proposes a batch of q
    candidates, one is selected via 'llm' (Selector agent), 'random', or
    'top_acquisition' (control conditions), then the selected candidate
    goes through the same Proposer->Critic->Verifier deliberation as the
    plain PCV method.
    """
    candidatos_tensor = bench.propose_batch(X_bo, Y_bo, bounds, q=q, seed=iteracion)

    if modo_seleccion == 'llm':
        candidatos_dict_list = [bench.tensor_to_dict(candidatos_tensor[i]) for i in range(q)]
        idx, selection_reasoning = bench.call_selector(candidatos_dict_list, historial, seed=iteracion)
    elif modo_seleccion == 'random':
        rng = __import__('numpy').random.default_rng(iteracion)
        idx = int(rng.integers(0, q))
        selection_reasoning = 'Random selection (control condition)'
    elif modo_seleccion == 'top_acquisition':
        valores = bench.compute_acquisition_values(candidatos_tensor, X_bo, Y_bo, bounds)
        idx = __import__('torch').argmax(valores).item()
        selection_reasoning = f'Highest acquisition value (control condition, value={valores[idx].item():.3f})'
    else:
        raise ValueError(f"Unknown modo_seleccion: {modo_seleccion}")

    candidato_seleccionado = candidatos_tensor[idx].unsqueeze(0)

    resultado = bench.run_pcv_deliberation(
        candidato_seleccionado, X_bo, Y_bo, historial,
        bounds=bounds, max_rechazos=max_rechazos, iteracion=iteracion, verbose=verbose
    )
    resultado['fuente'] = f"batch_{modo_seleccion}"
    resultado['selection_reasoning'] = selection_reasoning
    resultado['batch_index_selected'] = idx

    return resultado


def run_one_iteration(method, X_bo, Y_bo, historial, iteracion, seed_base):
    """
    Dispatch a single BO iteration to the correct method-specific function.
    A distinct-but-reproducible sub-seed is derived per iteration
    (seed_base * 1000 + iteracion) so torch.manual_seed doesn't reset to the
    same global RNG state on every one of the N_ITER iterations within a run.
    """
    iter_seed = seed_base * 1000 + iteracion

    if method == "bo_puro":
        return bench.run_bo_only_iteration(X_bo, Y_bo, bounds=bench.bounds, seed=iter_seed)

    elif method == "single_agent":
        return bench.run_single_agent_iteration(X_bo, Y_bo, historial, bounds=bench.bounds, seed=iter_seed)

    elif method == "multiagente":
        return run_pcv_iteration(
            X_bo, Y_bo, historial, bounds=bench.bounds, max_rechazos=3, iteracion=iteracion, verbose=False
        )

    elif method.startswith("batch_"):
        selection_mode = method.replace("batch_", "")
        return run_batch_iteration(
            X_bo, Y_bo, historial, bounds=bench.bounds, q=4,
            modo_seleccion=selection_mode, max_rechazos=3, iteracion=iteracion, verbose=False
        )

    else:
        raise ValueError(f"Unknown method: {method}")

## Seeds loop: experimental configuration and main run

Shared, fixed seed set, reused identically across every method (paired
comparisons). The six conditions below are the "core grid": batch's three
sub-modes (llm / random / top_acquisition) are treated as separate arms, to
compare the LLM selector against its two controls.

In [ ]:
SEEDS = list(range(10))  # [0, 1, ..., 9]
N_ITER = 30              # BO iterations to run after the initial Sobol design, per (method, seed)
METHODS = [
    "bo_puro",
    "single_agent",
    "multiagente",
    "batch_llm",
    "batch_random",
    "batch_top_acquisition",
]

# Flat list of records: one dict per (method, seed, iteration), so it
# converts directly into a pandas DataFrame for the metrics/plotting phase.
all_results = []

In [ ]:
import time
import pickle
import torch

CHECKPOINT_PATH = "seeds_loop_checkpoint_qwen30b.pkl"

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")
    X_init, Y_init, history_init = bench.generate_initial_design(seed)

    for method in METHODS:
        print(f"  -- Method: {method}")
        X_bo, Y_bo, historial = bench.clone_run_state(X_init, Y_init, history_init)
        t_start = time.time()

        for iteracion in range(1, N_ITER + 1):
            resultado = run_one_iteration(method, X_bo, Y_bo, historial, iteracion, seed_base=seed)

            x_final = resultado['x']
            y_final = resultado['y']

            # Grow THIS method's own X_bo/Y_bo/historial, so the next
            # iteration's GP is fit on everything seen so far.
            X_bo = torch.cat([X_bo, x_final])
            Y_bo = torch.cat([Y_bo, y_final])
            historial.append({
                'descriptors': bench.tensor_to_dict(x_final),
                'yield': y_final.item(),
            })

            all_results.append({
                'seed': seed,
                'method': method,
                'iteracion': iteracion,
                'yield': y_final.item(),
                'best_so_far': Y_bo.max().item(),
                'fuente': resultado['fuente'],
                'rechazos': resultado['rechazos'],
                'reasoning': resultado['reasoning'],
                'log_rechazos': resultado['log_rechazos'],
                'selection_reasoning': resultado.get('selection_reasoning'),
                'batch_index_selected': resultado.get('batch_index_selected'),
            })

        elapsed = time.time() - t_start
        print(f"     done in {elapsed/60:.1f} min, final best-so-far: {Y_bo.max().item():.2f}%")

        # Checkpoint after every (seed, method) run finishes -- this loop
        # can run for hours, we don't want to lose everything to a crash.
        with open(CHECKPOINT_PATH, "wb") as f:
            pickle.dump(all_results, f)

## Post-hoc analysis

A condensed subset of the original notebook's exploratory analysis cells --
sanity checks and the descriptor-keyword-mention count that is the source
of the (paper-reported) values hardcoded into Figure 12 in
`scripts/generate_figures.py`. None of this is required to reproduce the
paper's figures/tables (`scripts/generate_figures.py` does that directly
from the checkpoint files); it is kept for interpretability and provenance.

In [ ]:
import pandas as pd
import numpy as np

df_run = pd.DataFrame(all_results)
print(f"Total records: {len(df_run)}")
print(f"Expected: {len(SEEDS)} seeds x {len(METHODS)} methods x {N_ITER} iterations = {len(SEEDS) * len(METHODS) * N_ITER}")
print(f"\nAny missing/NaN yields? {df_run['yield'].isna().sum()}")
print("\nSource counts (fuente):")
print(df_run['fuente'].value_counts())
print("\nFallback rate per method (this run):")
df_run['fell_back'] = df_run['fuente'].isin(['bo_fallback'])
print(df_run.groupby('method')['fell_back'].mean())

In [ ]:
import re
from collections import Counter

DESCRIPTOR_KEYWORDS = {
    'MolWt': r'\bMolWt\b|\bmolecular weight\b',
    'TPSA': r'\bTPSA\b|\bpolar surface area\b',
    'MolLogP': r'\bMolLogP\b|\blogP\b|\blipophilicity\b',
    'NumHDonors': r'\bNumHDonors\b|\bH.?bond donor',
    'NumHAcceptors': r'\bNumHAcceptors\b|\bH.?bond acceptor',
    'NumRotatableBonds': r'\bNumRotatableBonds\b|\brotatable bond',
    'MaxPartialCharge': r'\bMaxPartialCharge\b|\bpartial charge\b',
    'MinPartialCharge': r'\bMinPartialCharge\b',
}

def count_mentions(text_series):
    counts = Counter()
    total_texts = 0
    for text in text_series.dropna():
        total_texts += 1
        for descriptor, pattern in DESCRIPTOR_KEYWORDS.items():
            if re.search(pattern, text, re.IGNORECASE):
                counts[descriptor] += 1
    return counts, total_texts

print("=== selection_reasoning (batch_llm) ===")
sel_texts = df_run[df_run['method'] == 'batch_llm']['selection_reasoning']
counts, total = count_mentions(sel_texts)
for descriptor in DESCRIPTOR_KEYWORDS:
    n = counts.get(descriptor, 0)
    if total:
        print(f"{descriptor}: {n}/{total} texts ({100*n/total:.1f}%)")

print("\n=== reasoning (multiagente) ===")
reason_texts = df_run[df_run['method'] == 'multiagente']['reasoning']
counts2, total2 = count_mentions(reason_texts)
for descriptor in DESCRIPTOR_KEYWORDS:
    n = counts2.get(descriptor, 0)
    if total2:
        print(f"{descriptor}: {n}/{total2} texts ({100*n/total2:.1f}%)")